# Experiment — HotpotQA
Cross-dataset replication: 3 models × 3 prompt types × {r=0, r=1} on HotpotQA (N=50).
Results saved to `results/hotpotqa_results.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
import pandas as pd
from pathlib import Path

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

MODELS       = cfg["models"]["available"]
PROMPT_TYPES = ["standard_qa", "cot_qa", "vigilant_qa"]
N            = cfg["evaluation"]["n_examples"]
SEED         = cfg["seed"]
RESULTS_DIR  = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Models: {MODELS}")
print(f"Prompt types: {PROMPT_TYPES}")
print(f"N: {N}  seed: {SEED}  k: {cfg['retrieval']['k']}")

In [ ]:
from src.data.download_hotpotqa import download
from src.data.hotpotqa_loader import load_hotpotqa

dev_path = Path("..") / cfg["dataset"]["hotpotqa_dev"]
download(target=dev_path)
examples = load_hotpotqa(str(dev_path), max_examples=N)
print(f"Loaded {len(examples)} HotpotQA examples")

In [ ]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever

emb_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])
print(f"k={cfg['retrieval']['k']}")

In [ ]:
from src.data.hotpotqa_loader import load_hotpotqa
from src.evaluation import qa_scorer
from src.generation.llm_client import HuggingFaceClient

examples_poisoned = load_hotpotqa("../" + cfg["dataset"]["hotpotqa_dev_poisoned"], max_examples=N)
print(f"Clean examples   : {len(examples)}")
print(f"Poisoned examples: {len(examples_poisoned)}")

records = []

for model in MODELS:
    print(f"\n{'='*60}\nModel: {model}\n{'='*60}")
    llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
    llm = HuggingFaceClient(
        model=model,
        temperature=cfg["models"]["temperature"],
        cache_dir=llm_cache,
    )
    with embedder, llm:
        for prompt_type in PROMPT_TYPES:
            print(f"  prompt={prompt_type}  r=0", end="")
            m0 = qa_scorer.run(
                examples=examples, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 0, **m0})
            print(f" EM={m0['exact_match']:.3f}  r=1", end="")
            m1 = qa_scorer.run(
                examples=examples_poisoned, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 1, **m1})
            print(f" EM={m1['exact_match']:.3f}")

print("\nSweep complete.")

In [ ]:
df = pd.DataFrame(records)
out_path = RESULTS_DIR / "hotpotqa_results.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")
df